In [0]:
import os
import sys
import shutil
from pathlib import Path
import json

# Force local file system synchronization
os.sync()

# Absolute workspace configuration
ROOT_DIR = Path("/Workspace/Repos/logi@openhealthagents.org/alphaesai/ClaimsProcessing")
sys.path.append(str(ROOT_DIR))

In [0]:
def move_file(src_path: Path, target_dir: Path) -> Path:
    """Moves a file to a target directory cleanly, ensuring the directory exists."""
    target_dir.mkdir(parents=True, exist_ok=True)
    target_path = target_dir / src_path.name
    
    # If target file exists, remove it first to allow safe overwrite/re-runs
    if target_path.exists():
        target_path.unlink()
        
    shutil.move(str(src_path), str(target_path))
    return target_path

In [0]:
def process_single_csv(incoming_file_path: Path, base_source_dir: Path, catalog_path: str) -> str:
    """Ingests a CSV file into the Bronze Delta table and advances directory states."""
    active_file_path = incoming_file_path
    try:
        if not active_file_path.exists():
            raise FileNotFoundError(f"Input file missing: {active_file_path}")
        
        # Move file to inprogress state
        active_file_path = move_file(active_file_path, base_source_dir / "inprogress")
        print(f"Processing data from: {active_file_path}")

        # Core Bronze Ingestion
        df = spark.read.format("csv") \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .load(str(active_file_path))
            
        # Deduplicate and trigger an immediate action to force data loading
        df = df.dropDuplicates()
        
        # Write to target catalog table
        df.write.format("delta").mode("overwrite").saveAsTable(catalog_path)
        print(f"Successfully wrote data to Bronze table: {catalog_path}")

        # --- FIX: Clear Spark cache to drop the file read-lock ---
        spark.catalog.clearCache()
        
        # Move file to processed state safely
        try:
            move_file(active_file_path, base_source_dir / "processed")
        except Exception as file_err:
            print(f"Warning: File processing succeeded but file move delayed: {file_err}")
            # Fallback copy-and-delete if direct move is locked
            target_dir = base_source_dir / "processed"
            target_dir.mkdir(parents=True, exist_ok=True)
            shutil.copy(str(active_file_path), str(target_dir / active_file_path.name))
            active_file_path.unlink()

        return active_file_path.name
        
    except Exception as e:
        print(f"Failed processing {active_file_path.name}: {e}")
        if active_file_path.exists():
            try:
                move_file(active_file_path, base_source_dir / "failed")
            except:
                pass
        raise

In [0]:
def trigger_silver_notebooks(client_container: str, config_path: str):
    """Triggers the Silver layer processing notebook with dynamic parameter payloads."""
    silver_notebook_path = f"{ROOT_DIR}/DimQualityEvent/Silver/Notebooks/GenericSubGroupProcessing"
    
    try:
        print("\n=== Triggering GenericSubGroupProcessing (Silver Layer) ===")
        dbutils.notebook.run(
            silver_notebook_path, 
            600, 
            {
                "ClientContainer": client_container,
                "SubGroupConfigPath": config_path
            }
        )
        print("Silver layer processing completed successfully.")
    except Exception as e:
        print(f"Silver layer processing failed: {e}")
        raise

In [0]:
def trigger_gold_notebooks(client_container: str, config_path: str):
    """Triggers the Gold layer processing notebook with dynamic parameter payloads."""
    gold_notebook_path = f"{ROOT_DIR}/DimQualityEvent/Gold/Notebooks/GenericSubGroupProcessing"
    
    try:
        print("\n=== Triggering GenericSubGroupProcessing (Gold Layer) ===")
        dbutils.notebook.run(
            gold_notebook_path, 
            600, 
            {
                "ClientContainer": client_container,
                "SubGroupConfigPath": config_path
            }
        )
        print("Gold layer processing completed successfully.")
    except Exception as e:
        print(f"Gold layer processing failed: {e}")
        raise

In [0]:
def main():
    # File routing setups
    base_source_dir = ROOT_DIR / "source/DimQualityEvent"
    pending_dir = base_source_dir / "pending"
    
    # Metadata and Pipeline Configurations
    client_container = "claimsprocessing"
    catalog_path = f"{client_container}.qe_bronze.quality_measures"
    silver_config_path = f"{ROOT_DIR}/DimQualityEvent/Silver/Schema/GapsInCare.json"
    gold_config_path = f"{ROOT_DIR}/DimQualityEvent/Gold/Schema/dimQualityEvent.json"
    
    if not pending_dir.exists():
        print(f"Pending directory does not exist: {pending_dir}")
        return
        
    incoming_files = [f for f in pending_dir.iterdir() if f.is_file() and f.suffix.lower() == '.csv' and not f.name.startswith('.')]
    if not incoming_files:
        print("No CSV files found to process.")
        return

    processed_filenames = []
    print(f"============================================================\nMAIN QUALITY EVENT EXECUTION STARTED\n============================================================")
    
    for file_path in incoming_files:
        try:
            filename = process_single_csv(file_path, base_source_dir, catalog_path)
            processed_filenames.append(filename)
        except Exception as e:
            print(f"Skipping file processing for {file_path.name} due to failure: {e}")

    if not processed_filenames:
        print("No files were successfully committed to Bronze. Execution halted.")
        return

    try:
        # Step 1: Run Silver processing loop
        trigger_silver_notebooks(client_container, silver_config_path)
        
        # Step 2: Run Gold processing loop
        trigger_gold_notebooks(client_container, gold_config_path)
        
        print("\nPipeline execution sequence completed successfully (Bronze → Silver → Gold).")
    except Exception as e:
        print(f"Downstream orchestrator execution failed: {e}")
        raise

if __name__ == "__main__":
    main()

In [0]:
# Verification cell - Run this to check current data status
print("=" * 70)
print("   QUALITYEVENT PIPELINE STATUS")
print("=" * 70)

# Check if tables exist and show row counts
layers = [
    ("Bronze", "claimsprocessing.qe_bronze.quality_measures"),
    ("Silver", "claimsprocessing.qe_silver.silver_gapsincare"),
    ("Gold", "claimsprocessing.qe_gold.dimQualityEvent")
]

for layer_name, table_name in layers:
    try:
        count = spark.table(table_name).count()
        print(f"\n✓ {layer_name}: {table_name}")
        print(f"  Rows: {count}")
        
        # Show last modified time
        history = spark.sql(f"DESCRIBE HISTORY {table_name} LIMIT 1").collect()
        if history:
            print(f"  Last Modified: {history[0]['timestamp']}")
    except Exception as e:
        print(f"\n⚠ {layer_name}: {table_name}")
        print(f"  Status: Table does not exist or is empty")

print("\n" + "=" * 70)